# Simple base-rate merged results

Load multi-model results from downloaded Kaggle Benchmarks runs, or from a local merged CSV.

Each row has **`score`** (`true`/`false`): for `scepticism_required=false` rows, whether the answer matches normative **P(C|T)**; for `scepticism_required=true` (`implausible_c_d`, `implausible_t`), whether it matches **`scepticism_score_target`** (keyed **H**). **`path_c_confusion`** flags answers matching **P(T|C)** (the inverse-conditional lure).

**Kaggle (all evaluated models):** after `kaggle auth login`:

```powershell
python -m kaggle benchmarks tasks download simple-rate-normative-accuracy `
  -o data/kaggle_runs/simple-rate-normative-accuracy
```

Or: `python scripts/export_simple_rate_kaggle_results.py --download`

Set `LOAD_FROM_KAGGLE = True` in the next cell (default).

In [78]:
from pathlib import Path
import importlib
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "simple").is_dir():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Reload so notebook picks up merge changes without a full kernel restart.
import benchmarks.kaggle_runs as kaggle_runs
import benchmarks.simple_rate as simple_rate

importlib.reload(simple_rate)
importlib.reload(kaggle_runs)

from benchmarks.kaggle_runs import (
    DEFAULT_SIMPLE_RATE_TASK_SLUG,
    download_task_runs,
    filter_run_rows_to_benchmark,
    find_run_json_files,
    load_base_rate_run_rows_from_tree,
)
from benchmarks.simple_rate import load_benchmark_rows, merge_run_results

print("benchmarks.kaggle_runs:", kaggle_runs.__file__)

LOAD_FROM_KAGGLE = True
DOWNLOAD_KAGGLE_RUNS = False  # True to refresh via Kaggle CLI before loading
KAGGLE_TASK_SLUG = DEFAULT_SIMPLE_RATE_TASK_SLUG
KAGGLE_RUNS_DIR = ROOT / "data" / "kaggle_runs" / KAGGLE_TASK_SLUG
BENCHMARK_CSV = ROOT / "data" / "simple" / "benchmark.csv"
MERGED_DIR = ROOT / "data" / "simple"

if LOAD_FROM_KAGGLE:
    if DOWNLOAD_KAGGLE_RUNS:
        download_task_runs(KAGGLE_TASK_SLUG, KAGGLE_RUNS_DIR)
    if not KAGGLE_RUNS_DIR.is_dir():
        raise FileNotFoundError(
            f"Download directory not found: {KAGGLE_RUNS_DIR}\n"
            f"Run: python -m kaggle benchmarks tasks download {KAGGLE_TASK_SLUG} "
            f"-o {KAGGLE_RUNS_DIR}"
        )
    run_files = find_run_json_files(KAGGLE_RUNS_DIR)
    _, benchmark_rows = load_benchmark_rows(BENCHMARK_CSV)
    benchmark_ids = {row["example_id"] for row in benchmark_rows}
    run_rows = load_base_rate_run_rows_from_tree(KAGGLE_RUNS_DIR)
    n_run_rows_before = len(run_rows)
    run_rows = filter_run_rows_to_benchmark(
        run_rows,
        benchmark_example_ids=benchmark_ids,
    )
    n_stale = n_run_rows_before - len(run_rows)
    if n_stale:
        print(
            f"Skipped {n_stale} stale run row(s) with example_ids not in "
            f"{BENCHMARK_CSV.name}."
        )
    # Only actual Kaggle runs (no padded blanks for missing model×prompt pairs).
    merged_rows = merge_run_results(
        run_rows,
        benchmark_path=BENCHMARK_CSV,
        fill_missing=False,
    )
    df = pd.DataFrame(merged_rows)
    data_source = f"Kaggle runs ({len(run_files)} *.run.json under {KAGGLE_RUNS_DIR})"
else:
    merged_candidates = sorted(
        MERGED_DIR.glob("simple_merged_results*.csv"),
        key=lambda candidate: candidate.stat().st_mtime,
        reverse=True,
    )
    if not merged_candidates:
        raise FileNotFoundError(
            f"Missing merged results under {MERGED_DIR}. "
            "Set LOAD_FROM_KAGGLE=True or run benchmark/simple-benchmark.ipynb."
        )
    MERGED_CSV = merged_candidates[0]
    df = pd.read_csv(MERGED_CSV)
    data_source = str(MERGED_CSV)

if "score" in df.columns:
    df["score_value"] = df["score"].astype(str).str.lower().eq("true").astype(int)
else:
    raise KeyError("Merged data must include 'score'.")

if "parseable" in df.columns:
    df["parseable_bool"] = df["parseable"].astype(str).str.lower().eq("true")
else:
    df["parseable_bool"] = True

if "path_c_confusion" in df.columns:
    df["path_c_confusion_bool"] = (
        df["path_c_confusion"].astype(str).str.lower().eq("true")
    )
else:
    df["path_c_confusion_bool"] = False

df["score_true"] = df["score_value"].astype(bool)

VARIANT_ORDER = ["open_probs", "mc_numeric_probs", "mc_full_probs"]
INTERSECTION_SIZE_ORDER = ["0", "small", "medium", "large"]
PROBLEM_TYPE_ORDER = ["well_posed", "implausible_c_d", "implausible_t"]
SCEPTICISM_REQUIRED_ORDER = ["false", "true"]

print("Source:", data_source)
print("Rows:", len(df))
n_empty_loaded = int(
    (
        df["llm_response"].isna()
        | df["llm_response"].astype(str).str.strip().eq("")
    ).sum()
)
print("Empty llm_response at load:", n_empty_loaded, "/", len(df))
if LOAD_FROM_KAGGLE and n_empty_loaded:
    raise RuntimeError(
        "Padded empty rows detected — merge should use fill_missing=False. "
        "Check benchmarks/simple_rate.py and re-run this cell."
    )
if not LOAD_FROM_KAGGLE and n_empty_loaded:
    print(
        "Tip: stale padded CSV — set LOAD_FROM_KAGGLE=True or re-export with "
        "scripts/export_simple_rate_kaggle_results.py"
    )
print("Models:", sorted(df["model"].unique()))
print("Vignettes:", df["vignette_name"].nunique())
print(
    "Normative pass:",
    int(df["score_true"].sum()),
    "/",
    len(df),
    "| P(T|C) confusion:",
    int(df["path_c_confusion_bool"].sum()),
    "/",
    len(df),
)
df.head()

Source: Kaggle runs (534 *.run.json under c:\src2\sceptical-llms\data\kaggle_runs\simple-rate-normative-accuracy)
Rows: 360
Empty llm_response at load: 149 / 360
Models: ['anthropic/claude-haiku-4-5@20251001', 'anthropic/claude-opus-4-1@20250805', 'anthropic/claude-opus-4-8@default', 'anthropic/claude-sonnet-4@20250514', 'google/gemini-2.5-flash', 'google/gemini-3-flash-preview', 'google/gemini-3.5-flash', 'openai/gpt-5.5-2026-04-23']
Vignettes: 9
Normative pass: 104 / 360 | P(T|C) confusion: 58 / 360


,example_id,vignette_name,problem_type,intersection_size,response_type,has_statistics,variant,prompt,well_posed,normative,...,parsed_choice,parsed_confidence,scoring_type,parseable,score,path_c_confusion,score_value,parseable_bool,path_c_confusion_bool,score_true
0,ca_trump_voter__implausible_c_d__mc_full_probs,CA Trump voter,implausible_c_d,0,mc_full,true,mc_full_probs,"You are a statistical consultant. Your task is to estimate a conditional probability from the information below.\n\nAmong US registered voters, 98% are vote...",true,implausible,...,A,,mc_full,true,false,false,0,True,False,False
1,ca_trump_voter__implausible_c_d__mc_full_probs,CA Trump voter,implausible_c_d,0,mc_full,true,mc_full_probs,"You are a statistical consultant. Your task is to estimate a conditional probability from the information below.\n\nAmong US registered voters, 98% are vote...",true,implausible,...,,,mc_full,false,false,false,0,False,False,False
2,ca_trump_voter__implausible_c_d__mc_full_probs,CA Trump voter,implausible_c_d,0,mc_full,true,mc_full_probs,"You are a statistical consultant. Your task is to estimate a conditional probability from the information below.\n\nAmong US registered voters, 98% are vote...",true,implausible,...,,,mc_full,false,false,false,0,False,False,False
3,ca_trump_voter__implausible_c_d__mc_full_probs,CA Trump voter,implausible_c_d,0,mc_full,true,mc_full_probs,"You are a statistical consultant. Your task is to estimate a conditional probability from the information below.\n\nAmong US registered voters, 98% are vote...",true,implausible,...,,,mc_full,false,false,false,0,False,False,False
4,ca_trump_voter__implausible_c_d__mc_full_probs,CA Trump voter,implausible_c_d,0,mc_full,true,mc_full_probs,"You are a statistical consultant. Your task is to estimate a conditional probability from the information below.\n\nAmong US registered voters, 98% are vote...",true,implausible,...,A,,mc_full,true,false,false,0,True,False,False


In [79]:
df["empty_llm_response_bool"] = (
    df["llm_response"].isna()
    | df["llm_response"].astype(str).str.strip().eq("")
)
n_empty = int(df["empty_llm_response_bool"].sum())
print(f"Rows with empty llm_response: {n_empty} / {len(df)}")

Rows with empty llm_response: 149 / 360


## Empty llm_response

Counts of rows with missing or blank `llm_response`, by design factor (before any row filtering).

In [80]:
from IPython.display import Markdown, display


def empty_response_summary_table(
    group_col: str, *, order: list[str] | None = None
) -> pd.DataFrame:
    work = df.copy()
    if group_col == "scepticism_required":
        work[group_col] = work[group_col].astype(str).str.lower()
    grouped = work.groupby(group_col, observed=True)
    summary = pd.DataFrame(
        {
            "n": grouped.size(),
            "empty_llm_response": grouped["empty_llm_response_bool"].sum().astype(int),
        }
    )
    summary["empty_pct"] = (summary["empty_llm_response"] / summary["n"] * 100).round(1)
    if order is not None:
        summary = summary.reindex([value for value in order if value in summary.index])
    elif group_col == "model":
        summary = summary.sort_index()
    return summary


print(
    f"Empty llm_response: {int(df['empty_llm_response_bool'].sum())} / {len(df)} "
    f"({df['empty_llm_response_bool'].mean() * 100:.1f}%)"
)

for title, col, order in [
    ("variant", "variant", VARIANT_ORDER),
    ("problem_type", "problem_type", PROBLEM_TYPE_ORDER),
    ("scepticism_required", "scepticism_required", SCEPTICISM_REQUIRED_ORDER),
    ("intersection_size", "intersection_size", INTERSECTION_SIZE_ORDER),
    ("model", "model", None),
    ("vignette", "vignette_name", sorted(df["vignette_name"].unique())),
]:
    display(Markdown(f"### By {title}"))
    display(empty_response_summary_table(col, order=order))

Empty llm_response: 149 / 360 (41.4%)


### By variant

,n,empty_llm_response,empty_pct
variant,,,
open_probs,72,1,1.4
mc_numeric_probs,72,0,0.0
mc_full_probs,216,148,68.5


### By problem_type

,n,empty_llm_response,empty_pct
problem_type,,,
well_posed,216,50,23.1
implausible_c_d,72,48,66.7
implausible_t,72,51,70.8


...

,n,empty_llm_response,empty_pct
intersection_size,,,
0,200,81,40.5
small,40,18,45.0
medium,40,15,37.5
large,80,35,43.8


### By model

,n,empty_llm_response,empty_pct
model,,,
anthropic/claude-haiku-4-5@20251001,45,13,28.9
anthropic/claude-opus-4-1@20250805,45,27,60.0
anthropic/claude-opus-4-8@default,45,27,60.0
anthropic/claude-sonnet-4@20250514,45,27,60.0
google/gemini-2.5-flash,45,1,2.2
google/gemini-3-flash-preview,45,0,0.0
google/gemini-3.5-flash,45,27,60.0
openai/gpt-5.5-2026-04-23,45,27,60.0


### By vignette

,n,empty_llm_response,empty_pct
vignette_name,,,
CA Trump voter,40,15,37.5
college STEM work,40,15,37.5
covid vaccine (blue/red),40,15,37.5
diabetes insulin obese,40,17,42.5
discharged weapon (last year),40,18,45.0
english teacher humanities,40,18,45.0
healthcare employment,40,16,40.0
military overseas (federal pool),40,17,42.5
professional drivers speeding,40,18,45.0


In [81]:
def score_summary_table(group_col: str, *, order: list[str] | None = None) -> pd.DataFrame:
    """Counts, parseability mix, normative score, and P(T|C) confusion by group."""
    work = df.copy()
    work["score_miss"] = work["parseable_bool"] & (work["score_value"] == 0)
    work["unparseable_row"] = ~work["parseable_bool"]

    grouped = work.groupby(group_col, observed=True)
    summary = pd.DataFrame(
        {
            "n": grouped.size(),
            "score_true": grouped["score_value"].sum(),
            "score_false": grouped["score_miss"].sum(),
            "unparseable": grouped["unparseable_row"].sum(),
            "path_c_confusion": grouped["path_c_confusion_bool"].sum(),
        }
    )
    summary["score_pct"] = (grouped["score_value"].mean() * 100).round(1)
    summary["path_c_pct"] = (grouped["path_c_confusion_bool"].mean() * 100).round(1)

    if order:
        summary = summary.reindex([value for value in order if value in summary.index])
    return summary


score_summary_table("variant", order=VARIANT_ORDER)

,n,score_true,score_false,unparseable,path_c_confusion,score_pct,path_c_pct
variant,,,,,,,
open_probs,72,34,35,3,55,47.2,76.4
mc_numeric_probs,72,57,13,2,3,79.2,4.2
mc_full_probs,216,13,55,148,0,6.0,0.0


## By problem_type

In [82]:
score_summary_table(
    "problem_type",
    order=[value for value in PROBLEM_TYPE_ORDER if value in df["problem_type"].unique()],
)

,n,score_true,score_false,unparseable,path_c_confusion,score_pct,path_c_pct
problem_type,,,,,,,
well_posed,216,100,62,54,58,46.3,26.9
implausible_c_d,72,2,22,48,0,2.8,0.0
implausible_t,72,2,19,51,0,2.8,0.0


## By scepticism_required

In [83]:
score_summary_table(
    "scepticism_required",
    order=[
        value
        for value in SCEPTICISM_REQUIRED_ORDER
        if value in df["scepticism_required"].astype(str).str.lower().unique()
    ],
)

,n,score_true,score_false,unparseable,path_c_confusion,score_pct,path_c_pct
scepticism_required,,,,,,,
false,216,100,62,54,58,46.3,26.9
true,144,4,41,99,0,2.8,0.0


## By intersection_size

In [84]:
score_summary_table(
    "intersection_size",
    order=[value for value in INTERSECTION_SIZE_ORDER if value in df["intersection_size"].unique()],
)

,n,score_true,score_false,unparseable,path_c_confusion,score_pct,path_c_pct
intersection_size,,,,,,,
0,200,71,45,84,32,35.5,16.0
small,40,15,7,18,6,37.5,15.0
medium,40,5,19,16,8,12.5,20.0
large,80,13,32,35,12,16.2,15.0


In [85]:
score_summary_table(
    "vignette_name",
    order=sorted(df["vignette_name"].unique()),
)

,n,score_true,score_false,unparseable,path_c_confusion,score_pct,path_c_pct
vignette_name,,,,,,,
CA Trump voter,40,17,7,16,6,42.5,15.0
college STEM work,40,5,19,16,8,12.5,20.0
covid vaccine (blue/red),40,12,11,17,7,30.0,17.5
diabetes insulin obese,40,11,12,17,7,27.5,17.5
discharged weapon (last year),40,16,6,18,6,40.0,15.0
english teacher humanities,40,2,20,18,5,5.0,12.5
healthcare employment,40,15,9,16,7,37.5,17.5
military overseas (federal pool),40,11,12,17,6,27.5,15.0
professional drivers speeding,40,15,7,18,6,37.5,15.0


## By model

In [86]:
score_summary_table(
    "model",
    order=sorted(df["model"].unique()),
)

,n,score_true,score_false,unparseable,path_c_confusion,score_pct,path_c_pct
model,,,,,,,
anthropic/claude-haiku-4-5@20251001,45,17,15,13,8,37.8,17.8
anthropic/claude-opus-4-1@20250805,45,14,4,27,8,31.1,17.8
anthropic/claude-opus-4-8@default,45,15,3,27,9,33.3,20.0
anthropic/claude-sonnet-4@20250514,45,13,5,27,9,28.9,20.0
google/gemini-2.5-flash,45,8,34,3,10,17.8,22.2
google/gemini-3-flash-preview,45,21,24,0,9,46.7,20.0
google/gemini-3.5-flash,45,6,10,29,3,13.3,6.7
openai/gpt-5.5-2026-04-23,45,10,8,27,2,22.2,4.4


## Unparseable responses

Model and raw response for rows where the answer could not be parsed.

In [87]:
unparseable = df.loc[~df["parseable_bool"]].sort_values(["model", "example_id"])
print(f"{len(unparseable)} unparseable rows")

pd.set_option("display.max_colwidth", None)
unparseable[["model", "llm_response"]]

153 unparseable rows


,model,llm_response
120,anthropic/claude-haiku-4-5@20251001,
128,anthropic/claude-haiku-4-5@20251001,
160,anthropic/claude-haiku-4-5@20251001,
168,anthropic/claude-haiku-4-5@20251001,
176,anthropic/claude-haiku-4-5@20251001,
...,...,...
295,openai/gpt-5.5-2026-04-23,
303,openai/gpt-5.5-2026-04-23,
327,openai/gpt-5.5-2026-04-23,
335,openai/gpt-5.5-2026-04-23,


## `mc_numeric_probs` detail

MC options, parsed letter, normative letter, score, and whether the choice is the **P(T|C)** lure.

In [88]:
MC_NUMERIC_CHOICE_COLS = [f"option_{letter}_label" for letter in "abcde"]
MC_LURE_COLS = [f"option_{letter}_lure" for letter in "abcde"]


def format_mc_choices(row: pd.Series) -> str:
    parts = []
    for letter, label_col, lure_col in zip("ABCDE", MC_NUMERIC_CHOICE_COLS, MC_LURE_COLS):
        label = row.get(label_col)
        lure = row.get(lure_col)
        if pd.notna(label) and str(label).strip():
            lure_text = f" [{lure}]" if pd.notna(lure) and str(lure).strip() else ""
            parts.append(f"{letter}: {label}{lure_text}")
    return " | ".join(parts)


mc_numeric_probs = df[df["variant"] == "mc_numeric_probs"].copy()
mc_numeric_probs["choices_offered"] = mc_numeric_probs.apply(format_mc_choices, axis=1)

mc_numeric_probs_view = mc_numeric_probs[
    [
        "vignette_name",
        "choices_offered",
        "parsed_choice",
        "normative_choice",
        "p_t_given_c",
        "score",
        "score_value",
        "path_c_confusion",
        "answer_line",
    ]
].sort_values("vignette_name")

pd.set_option("display.max_colwidth", 160)
mc_numeric_probs_view

,vignette_name,choices_offered,parsed_choice,normative_choice,p_t_given_c,score,score_value,path_c_confusion,answer_line
24,CA Trump voter,A: About 58% [Bayes P(C|T)] | B: About 8% [P(T|C)*P(T|D) confusion] | C: About 31% [P(T|C) confusion] | D: About 27% [P(T|D) confusion] | E: About 5% [P(C) ...,A,A,0.31,true,1,false,A
25,CA Trump voter,A: About 58% [Bayes P(C|T)] | B: About 8% [P(T|C)*P(T|D) confusion] | C: About 31% [P(T|C) confusion] | D: About 27% [P(T|D) confusion] | E: About 5% [P(C) ...,A,A,0.31,true,1,false,A
26,CA Trump voter,A: About 58% [Bayes P(C|T)] | B: About 8% [P(T|C)*P(T|D) confusion] | C: About 31% [P(T|C) confusion] | D: About 27% [P(T|D) confusion] | E: About 5% [P(C) ...,A,A,0.31,true,1,false,A
27,CA Trump voter,A: About 58% [Bayes P(C|T)] | B: About 8% [P(T|C)*P(T|D) confusion] | C: About 31% [P(T|C) confusion] | D: About 27% [P(T|D) confusion] | E: About 5% [P(C) ...,A,A,0.31,true,1,false,A
28,CA Trump voter,A: About 58% [Bayes P(C|T)] | B: About 8% [P(T|C)*P(T|D) confusion] | C: About 31% [P(T|C) confusion] | D: About 27% [P(T|D) confusion] | E: About 5% [P(C) ...,,A,0.31,false,0,false,P(T | CA) = (0.021
...,...,...,...,...,...,...,...,...,...
347,professional drivers speeding,A: About 85% [Bayes P(C|T)] | B: About 0% [P(D) confusion] | C: About 16% [P(T|C) confusion] | D: About 10% [P(T|D) confusion] | E: About 2% [P(T|C)*P(T|D) ...,A,A,0.16,true,1,false,A
348,professional drivers speeding,A: About 85% [Bayes P(C|T)] | B: About 0% [P(D) confusion] | C: About 16% [P(T|C) confusion] | D: About 10% [P(T|D) confusion] | E: About 2% [P(T|C)*P(T|D) ...,A,A,0.16,true,1,false,A
349,professional drivers speeding,A: About 85% [Bayes P(C|T)] | B: About 0% [P(D) confusion] | C: About 16% [P(T|C) confusion] | D: About 10% [P(T|D) confusion] | E: About 2% [P(T|C)*P(T|D) ...,A,A,0.16,true,1,false,A
350,professional drivers speeding,A: About 85% [Bayes P(C|T)] | B: About 0% [P(D) confusion] | C: About 16% [P(T|C) confusion] | D: About 10% [P(T|D) confusion] | E: About 2% [P(T|C)*P(T|D) ...,A,A,0.16,true,1,false,A


## `open_probs` detail

Re-parse open responses and compare to normative **P(C|T)** and **P(T|C)**.

In [89]:
from benchmarks.base_rate import matches_scepticism_target, parse_open_response
from benchmarks.simple_rate import PATH_C_LURE_NAME, load_benchmark, matches_path_c_confusion

benchmark_items = load_benchmark()


def rescore_open_row(row: pd.Series) -> pd.Series:
    item = benchmark_items[row["example_id"]]
    parsed = parse_open_response(str(row["llm_response"]))
    return pd.Series(
        {
            "parsed_numbers": list(parsed.percent_candidates),
            "parsed_percent_rescored": parsed.percent,
            "parsed_answer_type_rescored": parsed.answer_type,
            "parseable_rescored": parsed.answer_type != "unparseable",
            "score_true": matches_scepticism_target(item, parsed),
            "path_c_confusion_rescored": matches_path_c_confusion(item, parsed),
            "p_t_given_c_pct": float(row["p_t_given_c"]) * 100,
        }
    )


def _csv_bool(series: pd.Series) -> pd.Series:
    return series.astype(bool).map(lambda value: "true" if value else "false")


open_probs = df[df["variant"] == "open_probs"].copy()
open_probs = open_probs.drop(columns=["score_true"], errors="ignore")
open_probs = pd.concat([open_probs, open_probs.apply(rescore_open_row, axis=1)], axis=1)

idx = open_probs.index
rescored_percent = pd.to_numeric(open_probs["parsed_percent_rescored"], errors="coerce")
df.loc[idx, "parsed_percent"] = rescored_percent.map(
    lambda value: "" if pd.isna(value) else f"{value:g}"
)
df.loc[idx, "score_true"] = open_probs["score_true"].astype(bool)
df.loc[idx, "score_value"] = open_probs["score_true"].astype(int)
if "score" in df.columns:
    df.loc[idx, "score"] = _csv_bool(open_probs["score_true"])
df.loc[idx, "path_c_confusion_bool"] = open_probs["path_c_confusion_rescored"].astype(bool)
if "path_c_confusion" in df.columns:
    df.loc[idx, "path_c_confusion"] = _csv_bool(open_probs["path_c_confusion_rescored"])

open_probs_view = open_probs[
    [
        "example_id",
        "vignette_name",
        "normative_percent",
        "normative_open",
        "p_t_given_c_pct",
        "parsed_numbers",
        "parsed_percent_rescored",
        "score_true",
        "path_c_confusion_rescored",
    ]
].sort_values("vignette_name")

print(
    "Rescored open_probs normative pass:",
    int(open_probs["score_true"].sum()),
    "/",
    len(open_probs),
    "| P(T|C) confusion:",
    int(open_probs["path_c_confusion_rescored"].sum()),
    "/",
    len(open_probs),
)
open_probs_view

Rescored open_probs normative pass: 34 / 72 | P(T|C) confusion: 55 / 72


,example_id,vignette_name,normative_percent,normative_open,p_t_given_c_pct,parsed_numbers,parsed_percent_rescored,score_true,path_c_confusion_rescored
32,ca_trump_voter__open_probs,CA Trump voter,57.9,58%,31.0,"[7.8, 4.9, 27.0, 31.0, 58.0, 58.1, 2.106, 1.5190000000000001, 3.6249999999999996, 58.10999999999999]",58.1100,True,True
33,ca_trump_voter__open_probs,CA Trump voter,57.9,58%,31.0,"[7.8, 4.9, 27.0, 31.0, 58.1, 12.7, 2.106, 1.5190000000000001, 3.6249999999999996]",3.6250,True,True
34,ca_trump_voter__open_probs,CA Trump voter,57.9,58%,31.0,"[7.8, 4.9, 27.0, 31.0, 58.0, 2.106, 1.5190000000000001, 3.6249999999999996, 58.099999999999994]",58.1000,True,True
35,ca_trump_voter__open_probs,CA Trump voter,57.9,58%,31.0,"[7.8, 4.9, 27.0, 31.0, 12.7, 58.0, 61.4, 38.6, 16.6, 12.0, 28.599999999999998, 57.99999999999999]",58.0000,True,True
36,ca_trump_voter__open_probs,CA Trump voter,57.9,58%,31.0,"[7.8, 4.9, 27.0, 31.0, 12.7, 2.106, 1.5190000000000001]",1.5190,False,True
...,...,...,...,...,...,...,...,...,...
355,professional_drivers_speeding__open_probs,professional drivers speeding,85.49,85%,16.0,"[1.3, 0.35, 16.0, 10.0, 100.0, 85.59, 86.0, 35.0, 0.35000000000000003, 0.208, 0.034999999999999996, 0.243]",0.2430,True,True
356,professional_drivers_speeding__open_probs,professional drivers speeding,85.49,85%,16.0,"[1.3, 0.35, 16.0, 10.0, 0.35000000000000003, 35.0, 0.208, 0.034999999999999996, 0.243, 85.5967]",85.5967,True,True
357,professional_drivers_speeding__open_probs,professional drivers speeding,85.49,85%,16.0,"[1.3, 0.35, 16.0, 10.0, 85.6, 35.0, 0.35000000000000003, 0.208, 0.034999999999999996, 0.243, 85.5967]",85.5967,True,True
358,professional_drivers_speeding__open_probs,professional drivers speeding,85.49,85%,16.0,"[1.3, 0.35, 35.0, 0.35000000000000003]",0.3500,False,False


## `open_probs` vs `mc_numeric_probs` vs normative

Side-by-side for all 10 vignettes. **Normative** = P(C|T) (`normative_percent`). **P(T|C)** = `p_t_given_c` (inverse-conditional lure).

In [90]:
from benchmarks.base_rate import parse_open_response, parse_response
from benchmarks.simple_rate import PATH_C_LURE_NAME
from scripts.build_base_rate_prompts import _load_overlap

OVERLAP_VIGNETTE_NAMES = {v.name for v in _load_overlap()}

items_meta = pd.read_csv(ROOT / "data" / "simple" / "items.csv")


def _label_percent(label: str) -> float | None:
    text = (label or "").strip()
    if not text.startswith("About "):
        return None
    try:
        return float(text.removeprefix("About ").removesuffix("%"))
    except ValueError:
        return None


def _format_mc_choices(row: pd.Series) -> str:
    parts = []
    for letter in "ABCDE":
        label = row.get(f"option_{letter.lower()}_label")
        if pd.notna(label) and str(label).strip():
            parts.append(f"{letter}: {label}")
    return " | ".join(parts)


comparison_rows: list[dict] = []
for vignette_name in sorted(df["vignette_name"].unique()):
    open_row = df[(df["vignette_name"] == vignette_name) & (df["variant"] == "open_probs")].iloc[0]
    mc_row = df[(df["vignette_name"] == vignette_name) & (df["variant"] == "mc_numeric_probs")].iloc[0]
    item_open = items_meta.loc[items_meta["example_id"] == open_row["example_id"]].iloc[0]
    item_mc = items_meta.loc[items_meta["example_id"] == mc_row["example_id"]].iloc[0]

    parsed_open = parse_open_response(str(open_row["llm_response"]))
    parsed_mc = parse_response(str(mc_row["llm_response"]), scoring_type="mc_numeric")
    mc_choice = parsed_mc.choice or ""
    mc_label = item_mc.get(f"option_{mc_choice.lower()}_label", "") if mc_choice else ""
    mc_lure = item_mc.get(f"option_{mc_choice.lower()}_lure", "") if mc_choice else ""
    mc_pct = _label_percent(str(mc_label))

    normative_pct = float(item_open["normative_percent"])
    path_c_pct = float(item_open["p_t_given_c"]) * 100
    open_pct = parsed_open.percent

    comparison_rows.append(
        {
            "vignette_name": vignette_name,
            "overlap": vignette_name in OVERLAP_VIGNETTE_NAMES,
            "normative_open": item_open["normative_open"],
            "normative_pct": normative_pct,
            "p_t_given_c_pct": path_c_pct,
            "open_parsed_pct": open_pct,
            "open_delta_vs_norm_pp": None if open_pct is None else open_pct - normative_pct,
            "open_score": bool(open_row.get("score_true", open_row.get("score_value", 0))),
            "open_path_c": bool(open_row.get("path_c_confusion_bool", False)),
            "mc_choices": _format_mc_choices(item_mc),
            "mc_choice": mc_choice,
            "mc_label": mc_label,
            "mc_lure": mc_lure,
            "mc_parsed_pct": mc_pct,
            "mc_delta_vs_norm_pp": None if mc_pct is None else mc_pct - normative_pct,
            "mc_score": bool(mc_row.get("score_true", mc_row.get("score_value", 0))),
            "mc_path_c": mc_lure == PATH_C_LURE_NAME or (
                mc_pct is not None and abs(mc_pct - path_c_pct) <= 0.5
            ),
            "normative_mc_letter": item_mc["normative_choice"],
        }
    )

open_vs_mc = pd.DataFrame(comparison_rows).sort_values("vignette_name")

print(
    "open_probs pass:",
    int(open_vs_mc["open_score"].sum()),
    "/",
    len(open_vs_mc),
    "| mc_numeric_probs pass:",
    int(open_vs_mc["mc_score"].sum()),
    "/",
    len(open_vs_mc),
    "| open P(T|C) confusion:",
    int(open_vs_mc["open_path_c"].sum()),
    "| mc P(T|C) confusion:",
    int(open_vs_mc["mc_path_c"].sum()),
)

COMPARISON_COLUMNS = [
    "vignette_name",
    "overlap",
    "normative_pct",
    "open_parsed_pct",
    "open_score",
    "mc_choices",
    "mc_choice",
    "mc_label",
    "mc_score",
]

pd.set_option("display.max_colwidth", 160)
display(open_vs_mc[COMPARISON_COLUMNS])

open_probs pass: 6 / 9 | mc_numeric_probs pass: 9 / 9 | open P(T|C) confusion: 8 | mc P(T|C) confusion: 0


,vignette_name,overlap,normative_pct,open_parsed_pct,open_score,mc_choices,mc_choice,mc_label,mc_score
0,CA Trump voter,False,57.900,58.110,True,A: About 58% | B: About 8% | C: About 31% | D: About 27% | E: About 5%,A,About 58%,True
1,college STEM work,True,23.850,16.490,False,A: About 24% | B: About 63% | C: About 74% | D: About 17% | E: About 85%,A,About 24%,True
2,covid vaccine (blue/red),False,66.200,66.090,True,A: About 66% | B: About 1% | C: About 8% | D: About 10% | E: About 19%,A,About 66%,True
3,diabetes insulin obese,True,42.810,1.488,True,A: About 43% | B: About 5% | C: About 16% | D: About 20% | E: About 3%,A,About 43%,True
4,discharged weapon (last year),False,77.270,77.590,True,A: About 77% | B: About 0% | C: About 13% | D: About 30%,A,About 77%,True
5,english teacher humanities,True,52.060,NaN,False,A: About 52% | B: About 69% | C: About 55% | D: About 0% | E: About 38%,A,About 52%,True
6,healthcare employment,False,9.091,6.534,True,A: About 9% | B: About 10% | C: About 32% | D: About 60% | E: About 54%,A,About 9%,True
7,military overseas (federal pool),False,38.350,38.800,True,A: About 38% | B: About 37% | C: About 58% | D: About 64% | E: About 20%,A,About 38%,True
8,professional drivers speeding,True,85.490,98.300,False,A: About 85% | B: About 0% | C: About 16% | D: About 10% | E: About 2%,A,About 85%,True


basically, whenever overlap is false, everything is correct; but when overlap is true, the open output is wrong, but the mc is correct.
COuld this be due to the MC being way to easy?

### Printable comparison

Per vignette: source probabilities from `items.csv` (P(C), P(D), P(T|C), P(T|D)), normative / open / MC answers, and full `mc_numeric_probs` prompt.

In [91]:
import re

from benchmarks.base_rate import parse_open_response, parse_response

benchmark_df = pd.read_csv(ROOT / "data" / "simple" / "benchmark.csv")


def mc_numeric_options_prompt(prompt: str) -> str:
    lines = [line.strip() for line in prompt.splitlines() if line.strip()]
    option_lines = [line for line in lines if re.match(r"^[A-E]\.\s", line)]
    return " | ".join(option_lines)


def format_source_ps(item_row: pd.Series) -> str:
    return " | ".join(
        [
            f"P(C)={float(item_row['p_c']):.6g}",
            f"P(D)={float(item_row['p_d']):.6g}",
            f"P(T|C)={float(item_row['p_t_given_c']):.6g}",
            f"P(T|D)={float(item_row['p_t_given_d']):.6g}",
        ]
    )


print_rows: list[dict] = []
for vignette_name in sorted(df["vignette_name"].unique()):
    open_row = df[(df["vignette_name"] == vignette_name) & (df["variant"] == "open_probs")].iloc[0]
    mc_row = df[(df["vignette_name"] == vignette_name) & (df["variant"] == "mc_numeric_probs")].iloc[0]
    item_open = items_meta.loc[items_meta["example_id"] == open_row["example_id"]].iloc[0]
    item_mc = items_meta.loc[items_meta["example_id"] == mc_row["example_id"]].iloc[0]
    bench_row = benchmark_df.loc[benchmark_df["example_id"] == mc_row["example_id"]].iloc[0]

    parsed_open = parse_open_response(str(open_row["llm_response"]))
    parsed_mc = parse_response(str(mc_row["llm_response"]), scoring_type="mc_numeric")
    mc_choice = parsed_mc.choice or ""
    mc_label = (
        str(item_mc.get(f"option_{mc_choice.lower()}_label", ""))
        if mc_choice
        else ""
    )
    full_prompt = str(bench_row["prompt"])

    print_rows.append(
        {
            "vignette_name": vignette_name,
            "source_ps": format_source_ps(item_open),
            "normative_pct": float(item_open["normative_percent"]),
            "p_t_given_c_pct": float(item_open["p_t_given_c"]) * 100,
            "open_parsed_pct": parsed_open.percent,
            "mc_label": f"{mc_choice} {mc_label}".strip(),
            "numeric_prompt": mc_numeric_options_prompt(full_prompt),
            "prompt": full_prompt,
        }
    )

print_table = pd.DataFrame(print_rows).sort_values("vignette_name")

print(f"{'vignette_name':<32} {'normative':>10} {'P(T|C)':>10} {'open':>10} {'MC label':>14}")
print("-" * 84)
for row in print_table.itertuples(index=False):
    open_pct = "—" if pd.isna(row.open_parsed_pct) else f"{row.open_parsed_pct:.4g}%"
    print(f"\n{row.vignette_name}")
    print(f"  source Ps:   {row.source_ps}")
    print(f"  normative:   {row.normative_pct:.4g}%  (P(C|T))")
    print(f"  P(T|C):      {row.p_t_given_c_pct:.4g}%  (inverse-conditional lure)")
    print(f"  open parsed: {open_pct}")
    print(f"  MC label:    {row.mc_label}")
    print(f"  numeric prompt: {row.numeric_prompt}")
    print("  prompt:")
    for line in row.prompt.splitlines():
        print(f"    {line}")

print_table.drop(columns=["prompt"])

vignette_name                     normative     P(T|C)       open       MC label
------------------------------------------------------------------------------------

CA Trump voter
  source Ps:   P(C)=0.0494 | P(D)=0.078 | P(T|C)=0.31 | P(T|D)=0.27
  normative:   57.9%  (P(C|T))
  P(T|C):      31%  (inverse-conditional lure)
  open parsed: 58.11%
  MC label:    A About 58%
  numeric prompt: A. About 58% | B. About 8% | C. About 31% | D. About 27% | E. About 5%
  prompt:
    You are a statistical consultant. Your task is to estimate a conditional probability from the information below.
    
    Among US registered voters, 7.8% are voters registered in Southern California and 4.9% are other California voters.
    Among voters registered in Southern California, 27% voted for Donald Trump in the 2024 presidential election; among other California voters, 31% voted for Donald Trump in the 2024 presidential election.
    
    What is the probability that a registered voter in California who 

,vignette_name,source_ps,normative_pct,p_t_given_c_pct,open_parsed_pct,mc_label,numeric_prompt
0,CA Trump voter,P(C)=0.0494 | P(D)=0.078 | P(T|C)=0.31 | P(T|D)=0.27,57.900,31.0,58.110,A About 58%,A. About 58% | B. About 8% | C. About 31% | D. About 27% | E. About 5%
1,college STEM work,P(C)=0.04644 | P(D)=0.17028 | P(T|C)=0.85 | P(T|D)=0.74,23.850,85.0,16.490,A About 24%,A. About 24% | B. About 63% | C. About 74% | D. About 17% | E. About 85%
2,covid vaccine (blue/red),P(C)=0.1917 | P(D)=0.0783 | P(T|C)=0.08 | P(T|D)=0.1,66.200,8.0,66.090,A About 66%,A. About 66% | B. About 1% | C. About 8% | D. About 10% | E. About 19%
3,diabetes insulin obese,P(C)=0.031866 | P(D)=0.053223 | P(T|C)=0.2 | P(T|D)=0.16,42.810,20.0,1.488,A About 43%,A. About 43% | B. About 5% | C. About 16% | D. About 20% | E. About 3%
4,discharged weapon (last year),P(C)=0.2992 | P(D)=0.132 | P(T|C)=0.003 | P(T|D)=0.002,77.270,0.3,77.590,A About 77%,A. About 77% | B. About 0% | C. About 13% | D. About 30%
5,english teacher humanities,P(C)=0.0011248 | P(D)=0.0012996 | P(T|C)=0.69 | P(T|D)=0.55,52.060,69.0,NaN,A About 52%,A. About 52% | B. About 69% | C. About 55% | D. About 0% | E. About 38%
6,healthcare employment,P(C)=0.011 | P(D)=0.099 | P(T|C)=0.54 | P(T|D)=0.6,9.091,54.0,6.534,A About 9%,A. About 9% | B. About 10% | C. About 32% | D. About 60% | E. About 54%
7,military overseas (federal pool),P(C)=0.14 | P(D)=0.204 | P(T|C)=0.58 | P(T|D)=0.64,38.350,58.0,38.800,A About 38%,A. About 38% | B. About 37% | C. About 58% | D. About 64% | E. About 20%
8,professional drivers speeding,P(C)=0.012816 | P(D)=0.00348 | P(T|C)=0.16 | P(T|D)=0.1,85.490,16.0,98.300,A About 85%,A. About 85% | B. About 0% | C. About 16% | D. About 10% | E. About 2%


## Optional: split by model when multiple LLMs are present

In [92]:
if df["model"].nunique() > 1:
    display(
        df.groupby(["model", "variant"], observed=True)["score_value"]
        .mean()
        .unstack("variant")
        .reindex(columns=VARIANT_ORDER)
        .round(3)
    )
    display(
        df.groupby(["model", "variant"], observed=True)["path_c_confusion_bool"]
        .mean()
        .unstack("variant")
        .reindex(columns=VARIANT_ORDER)
        .round(3)
    )
else:
    print("Single model in file — see tables above.")

variant,open_probs,mc_numeric_probs,mc_full_probs
model,,,
anthropic/claude-haiku-4-5@20251001,0.667,1.000,0.074
anthropic/claude-opus-4-1@20250805,0.667,0.889,0.000
anthropic/claude-opus-4-8@default,0.778,0.889,0.000
anthropic/claude-sonnet-4@20250514,0.556,0.889,0.000
google/gemini-2.5-flash,0.111,0.444,0.111
google/gemini-3-flash-preview,0.667,0.778,0.296
google/gemini-3.5-flash,0.000,0.667,0.000
openai/gpt-5.5-2026-04-23,0.333,0.778,0.000


variant,open_probs,mc_numeric_probs,mc_full_probs
model,,,
anthropic/claude-haiku-4-5@20251001,0.889,0.000,0.0
anthropic/claude-opus-4-1@20250805,0.889,0.000,0.0
anthropic/claude-opus-4-8@default,1.000,0.000,0.0
anthropic/claude-sonnet-4@20250514,1.000,0.000,0.0
google/gemini-2.5-flash,0.889,0.222,0.0
google/gemini-3-flash-preview,1.000,0.000,0.0
google/gemini-3.5-flash,0.222,0.111,0.0
openai/gpt-5.5-2026-04-23,0.222,0.000,0.0
